## 1. Hugging Face Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Check GPU

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 3. Configuration

#### this is sample configuration, you are free to modify any parameter to train the best performing model

In [3]:
import os
import random
import json
import time
import math
import zipfile
import shutil

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# You can use any of the 4 train files or all the files for pre-training the legal model
TRAIN_FILE = ("Phase_1_Pretraining/Train/train_acts.json",
             "Phase_1_Pretraining/Train/train_high_courts.json",
             "Phase_1_Pretraining/Train/train_supreme_court.json",
             "Phase_1_Pretraining/Train/train_tribunals.json")
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Training configuration
TRAIN_TOKENS = 4_000_000
VAL_TOKENS = 1_000_000
TRAIN_BLOCK_SIZE = 1024
EPOCHS = 1
SEED = 42
random.seed(SEED)
MODEL_CONTEXT_LENGTH = 1024
N_EMBD = 560
N_LAYER = 12
N_HEAD = 8

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 1.0
TOP_P = 0.95

# Kaggle output
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
MODEL_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "legal-gpt2-final"
)
PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "prediction.json"
)
SUBMISSION_ZIP = os.path.join(
    OUTPUT_DIR,
    "submission.zip"
)
LOSS_PLOT_PATH = os.path.join(
    OUTPUT_DIR,
    "training_validation_loss.png"
)
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)
print("Configuration loaded.")

Configuration loaded.


## 4. Load Tokenizer

In [ ]:
from collections import defaultdict, Counter
import heapq

SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class BytelevelBPE:

    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.specials = SPECIALS.copy()
        self.byte_vocab = [str(i) for i in range(256)]
        self.merges = []
        self.merge_ranks = {}
        self.vocab = []
        self.stoi = {}
        self.itos = {}
        # Maps each BPE symbol to the bytes represented by it
        self.symbol_to_bytes = {}
    
    # 1. WORD -> UTF-8 BYTE SYMBOLS

    def word_to_symbols(self, word):
        byte_seq = word.encode("utf-8")
        return [str(b) for b in byte_seq] + ["</w>"]

    # 2. BUILD WORD FREQUENCIES

    def build_word_freqs(self, corpus):
        word_counter = Counter()
        print("Building word frequencies...")
        for text in corpus:
            if not isinstance(text, str):
                continue

            word_counter.update(text.split())
        print(
            f"Unique whitespace-separated words: "
            f"{len(word_counter):,}"
        )
        # Convert each unique word to byte symbols only once
        word_freqs = {}

        for word, freq in word_counter.items():
            symbols = tuple(self.word_to_symbols(word))
            word_freqs[symbols] = (
                word_freqs.get(symbols, 0) + freq
            )

        return word_freqs


    # 3. PAIRS OCCURRING IN ONE WORD

    @staticmethod
    def get_pairs(symbols):
        return {
            (symbols[i], symbols[i + 1])
            for i in range(len(symbols) - 1)
        }

    # 4. COUNT PAIR OCCURRENCES IN ONE WORD

    @staticmethod
    def pair_counts_in_word(symbols):
        counts = Counter()
        for i in range(len(symbols) - 1):
            counts[
                (symbols[i], symbols[i + 1])
            ] += 1

        return counts

    # 5. MERGE A PAIR INSIDE ONE WORD

    @staticmethod
    def merge_symbols(symbols, pair, merged_symbol):
        new_symbols = []
        i = 0
        while i < len(symbols):
            if (
                i < len(symbols) - 1
                and symbols[i] == pair[0]
                and symbols[i + 1] == pair[1]
            ):
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1

        return tuple(new_symbols)

    # 6. TRAIN OPTIMIZED BPE

    def train(self, corpus):
        word_freqs = self.build_word_freqs(corpus)
        vocab = (
            self.specials
            + self.byte_vocab
            + ["</w>"]
        )
        seen = set(vocab)
        for b in range(256):
            self.symbol_to_bytes[str(b)] = bytes([b])
        self.symbol_to_bytes["</w>"] = b""

        words = []
        frequencies = []
        for symbols, freq in word_freqs.items():
            words.append(symbols)
            frequencies.append(freq)
        del word_freqs

        pair_counts = defaultdict(int)
        pair_to_words = defaultdict(set)

        print("Building initial pair statistics...")

        for word_id, symbols in enumerate(words):
            freq = frequencies[word_id]
            local_counts = self.pair_counts_in_word(symbols)
            for pair, count in local_counts.items():
                pair_counts[pair] += count * freq
                pair_to_words[pair].add(word_id)

        print(
            f"Initial unique pairs: "
            f"{len(pair_counts):,}"
        )

        # MAX HEAP
        #
        # Python has min-heap, so use negative frequency.
        #
        # We keep stale entries and validate them when popped.
        # This avoids rebuilding the heap after every merge.

        heap = []
        for pair, count in pair_counts.items():
            heapq.heappush(heap,(-count, pair))
        target_merges = max(0,self.vocab_size - len(vocab))
        print(f"Target merges: {target_merges:,}")
        while (len(vocab) < self.vocab_size and heap):
            while heap:
                neg_count, pair = heapq.heappop(heap)
                heap_count = -neg_count
                current_count = pair_counts.get(pair,0)
                # Ignore stale heap entries
                if (current_count == heap_count and current_count > 0):
                    break
            else:
                break

            max_freq = pair_counts.get(pair, 0)

            # Same stopping condition as before
            if max_freq <= 1:
                break

            # Create collision-free merged symbol
            merged_symbol = (f"<bpe_{len(self.merges)}>")

            # Byte representation of new symbol
            left_bytes = self.symbol_to_bytes[pair[0]]
            right_bytes = self.symbol_to_bytes[pair[1]]
            self.symbol_to_bytes[merged_symbol] = left_bytes + right_bytes

            self.merges.append(pair)
            self.merge_ranks[pair] = (len(self.merges) - 1,merged_symbol)
            if merged_symbol not in seen:
                seen.add(merged_symbol)
                vocab.append(merged_symbol)

            # ----------------------------------------------------
            # IMPORTANT OPTIMIZATION
            #
            # Only process words containing selected pair.
            # ----------------------------------------------------

            affected_word_ids = list(pair_to_words.get(pair, set()))
            changed_pairs = set()
            for word_id in affected_word_ids:
                old_symbols = words[word_id]
                freq = frequencies[word_id]
                # Pair may have disappeared because of an earlier
                # update involving this word
                if pair not in self.get_pairs(old_symbols):
                    continue
                
                # Statistics BEFORE merge
                old_counts = self.pair_counts_in_word(old_symbols)

                # Merge selected pair
                new_symbols = self.merge_symbols(old_symbols,pair,merged_symbol)
                if new_symbols == old_symbols:
                    continue

                # Statistics AFTER merge
                new_counts = self.pair_counts_in_word(new_symbols)

                # Save updated word
                words[word_id] = new_symbols

                all_pairs = (
                    set(old_counts)
                    | set(new_counts)
                )

                # Update only pairs affected by this word

                for p in all_pairs:

                    old_n = old_counts.get(p, 0)
                    new_n = new_counts.get(p, 0)
                    delta = (
                        new_n - old_n
                    ) * freq

                    if delta != 0:
                        pair_counts[p] += delta
                        changed_pairs.add(p)

                    # Update inverted index
                    if new_n > 0:
                        pair_to_words[p].add(word_id)
                    else:
                        pair_to_words[p].discard(word_id)

            # Selected pair should now be gone from affected words
            pair_to_words[pair].clear()

            # Push updated frequencies into heap
            for p in changed_pairs:
                count = pair_counts.get(p, 0)
                if count > 0:
                    heapq.heappush(heap,(-count, p))

        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.stoi = {
            symbol: i
            for i, symbol in enumerate(vocab)
        }

        self.itos = {
            i: symbol
            for symbol, i in self.stoi.items()
        }

        print("\nTokenizer training complete.")

        print(
            "Final vocabulary size:",
            self.vocab_size
        )

        print(
            "Number of merges:",
            len(self.merges)
        )

    # 7. APPLY BPE TO ONE WORD

    def apply_bpe(self, word):
        symbols = self.word_to_symbols(word)
        while len(symbols) > 1:
            best_pair = None
            best_rank = float("inf")
            best_merged_symbol = None

            for i in range(len(symbols) - 1):
                pair = (symbols[i],symbols[i + 1])
                merge_info = self.merge_ranks.get(pair)
                if merge_info is not None:
                    rank, merged_symbol = merge_info
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = pair
                        best_merged_symbol = merged_symbol

            if best_pair is None:
                break

            symbols = list(
                self.merge_symbols(
                    tuple(symbols),
                    best_pair,
                    best_merged_symbol
                )
            )

        return symbols

    # 8. ENCODE

    def encode(self,text,add_bos=False,add_eos=False):
        ids = []
        if add_bos:
            ids.append(
                self.stoi["<s>"]
            )

        for word in text.split():
            symbols = self.apply_bpe(word)
            for symbol in symbols:
                ids.append(
                    self.stoi.get(
                        symbol,
                        self.stoi["<unk>"]
                    )
                )

        if add_eos:
            ids.append(
                self.stoi["</s>"]
            )

        return ids

    # 9. DECODE

    def decode(self, ids):

        words = []
        current_bytes = bytearray()

        for token_id in ids:
            symbol = self.itos.get(token_id)
            if symbol is None:
                continue
            if symbol in self.specials:
                continue

            # Determine whether this token ends a word
            ends_word = self._symbol_ends_word(symbol)
            byte_value = self.symbol_to_bytes.get(symbol, b"")
            current_bytes.extend(byte_value)

            if ends_word:
                try:
                    word = bytes(current_bytes).decode("utf-8")
                    words.append(word)
                except UnicodeDecodeError:
                    words.append("<unk>")
                current_bytes = bytearray()

        # Remaining bytes
        if current_bytes:
            try:
                words.append(bytes(current_bytes).decode("utf-8"))
            except UnicodeDecodeError:
                words.append("<unk>")

        return " ".join(words)

    # 10. CHECK WHETHER SYMBOL CONTAINS </w> AT END

    def _symbol_ends_word(self, symbol):
        if symbol == "</w>":
            return True
        # Find merge that created this symbol
        for pair, (_, merged) in self.merge_ranks.items():
            if merged == symbol:
                return self._symbol_ends_word(
                    pair[1]
                )

        return False

    # 11. VOCAB SIZE

    def get_vocab_size(self):
        return len(self.vocab)

## 5. Download Training Data From Hugging Face

In [5]:
from huggingface_hub import hf_hub_download

train_file_path_acts = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE[0],
    repo_type="dataset",
    token=HF_TOKEN
)

train_file_path_high_courts = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE[1],
    repo_type="dataset",
    token=HF_TOKEN
)

train_file_path_supreme_court = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE[2],
    repo_type="dataset",
    token=HF_TOKEN
)

train_file_path_tribunals = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE[3],
    repo_type="dataset",
    token=HF_TOKEN
)

print("Training file:")
print(train_file_path_acts)

print("Training file:")
print(train_file_path_high_courts)

print("Training file:")
print(train_file_path_supreme_court)

print("Training file:")
print(train_file_path_tribunals)

Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_supreme_(…): reconstructing file:   0%|          |  0.00B / 1.32GB            

Phase_1_Pretraining/Train/train_supreme_(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

Training file:
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json
Training file:
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_high_courts.json
Training file:
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json
Training file:
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json


In [6]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [7]:
source_paths = {
    "acts": train_file_path_acts,
    "high_courts": train_file_path_high_courts,
    "supreme_court": train_file_path_supreme_court,
    "tribunals": train_file_path_tribunals,
}

In [8]:
import os
import json
import random
import hashlib
import numpy as np

SOURCE_RATIOS = {
    "acts": 1,
    "high_courts": 3,
    "supreme_court": 2,
    "tribunals": 2,
}

TOKENIZER_WORD_BUDGET = 4_000_000
TRAIN_TOKEN_BUDGET = 800_000_000
VAL_TOKEN_BUDGET = 8_000_000  # 500,000 in your configuration

# Use your chosen document-level validation fraction.
VALIDATION_FRACTION = 0.05

def allocate_budget(total):
    ratio_sum = sum(SOURCE_RATIOS.values())

    budgets = {
        name: total * ratio // ratio_sum
        for name, ratio in SOURCE_RATIOS.items()
    }

    # Handle rounding for budgets not divisible by five.
    remainder = total - sum(budgets.values())
    for name in list(budgets)[:remainder]:
        budgets[name] += 1

    return budgets

word_budgets = allocate_budget(TOKENIZER_WORD_BUDGET)
train_budgets = allocate_budget(TRAIN_TOKEN_BUDGET)
val_budgets = allocate_budget(VAL_TOKEN_BUDGET)

print("Tokenizer words:", word_budgets)
print("Training tokens:", train_budgets)
print("Validation tokens:", val_budgets)

Tokenizer words: {'acts': 500000, 'high_courts': 1500000, 'supreme_court': 1000000, 'tribunals': 1000000}
Training tokens: {'acts': 100000000, 'high_courts': 300000000, 'supreme_court': 200000000, 'tribunals': 200000000}
Validation tokens: {'acts': 1000000, 'high_courts': 3000000, 'supreme_court': 2000000, 'tribunals': 2000000}


In [9]:
%pip install -q ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/150.2 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.2/150.2 kB 6.0 MB/s eta 0:00:00


Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
import json
import hashlib
import sqlite3
import ijson

SPLIT_DIR = os.path.join(OUTPUT_DIR, "document_splits")
os.makedirs(SPLIT_DIR, exist_ok=True)

DEDUP_PATH = os.path.join(SPLIT_DIR, "dedup.sqlite")

# Rebuild from scratch when this cell is rerun.
if os.path.exists(DEDUP_PATH):
    os.remove(DEDUP_PATH)

connection = sqlite3.connect(DEDUP_PATH)
connection.execute(
    "CREATE TABLE seen (digest BLOB PRIMARY KEY) WITHOUT ROWID"
)


class JsonlTexts:
    """Reads texts from disk; each iteration starts from the beginning."""

    def __init__(self, path):
        self.path = path

    def __iter__(self):
        with open(self.path, "r", encoding="utf-8") as f:
            for line in f:
                yield json.loads(line)


train_sources = {}
val_sources = {}
split_counts = {}

try:
    for name, source_path in source_paths.items():
        train_path = os.path.join(SPLIT_DIR, f"{name}_train.jsonl")
        val_path = os.path.join(SPLIT_DIR, f"{name}_val.jsonl")

        counts = {
            "train": 0,
            "validation": 0,
            "invalid": 0,
            "duplicates": 0,
        }

        print(f"\nStreaming {name}...", flush=True)

        with (
            open(source_path, "rb") as source,
            open(train_path, "w", encoding="utf-8") as train_out,
            open(val_path, "w", encoding="utf-8") as val_out,
        ):
            # Verify the top-level structure without loading the file.
            first = source.read(1)
            while first and first.isspace():
                first = source.read(1)

            if first != b"[":
                raise ValueError(f"{name}: expected a top-level JSON list.")

            source.seek(0)

            for index, record in enumerate(
                ijson.items(source, "item"), start=1
            ):
                text = (
                    record.get("text")
                    if isinstance(record, dict)
                    else None
                )

                if not isinstance(text, str) or not text.strip():
                    counts["invalid"] += 1
                    continue

                text = " ".join(text.split())
                digest = hashlib.sha256(text.encode("utf-8")).digest()

                cursor = connection.execute(
                    "INSERT OR IGNORE INTO seen VALUES (?)",
                    (digest,),
                )

                if cursor.rowcount == 0:
                    counts["duplicates"] += 1
                    continue

                # Deterministic document-level split.
                split_digest = hashlib.sha256(
                    str(SEED).encode("ascii") + digest
                ).digest()

                fraction = (
                    int.from_bytes(split_digest[:8], "big") / (1 << 64)
                )

                if fraction < VALIDATION_FRACTION:
                    destination = val_out
                    counts["validation"] += 1
                else:
                    destination = train_out
                    counts["train"] += 1

                destination.write(
                    json.dumps(text, ensure_ascii=False) + "\n"
                )

                if index % 10_000 == 0:
                    connection.commit()

        connection.commit()

        train_sources[name] = JsonlTexts(train_path)
        val_sources[name] = JsonlTexts(val_path)
        split_counts[name] = counts

        print(name, counts, flush=True)

finally:
    connection.close()


Streaming acts...


acts {'train': 17775, 'validation': 956, 'invalid': 4, 'duplicates': 91}



Streaming high_courts...


high_courts {'train': 2297028, 'validation': 121220, 'invalid': 0, 'duplicates': 118490}



Streaming supreme_court...


supreme_court {'train': 52572, 'validation': 2789, 'invalid': 21, 'duplicates': 167}



Streaming tribunals...


tribunals {'train': 820769, 'validation': 43429, 'invalid': 0, 'duplicates': 4934}


In [11]:
def select_word_budget(documents, target_words, source_name):
    selected = []
    collected = 0

    for text in documents:
        remaining = target_words - collected

        if remaining == 0:
            break

        words = text.split()

        if len(words) <= remaining:
            selected.append(text)
            collected += len(words)
        else:
            selected.append(" ".join(words[:remaining]))
            collected += remaining

    if collected != target_words:
        raise ValueError(
            f"{source_name}: needs {target_words:,} words, "
            f"but only {collected:,} are available in its training split."
        )

    return selected

tokenizer_corpus = []

for name, budget in word_budgets.items():
    selected = select_word_budget(
        train_sources[name], budget, name
    )
    tokenizer_corpus.extend(selected)

    print(f"{name}: {budget:,} words")
    del selected

random.Random(SEED).shuffle(tokenizer_corpus)

print("Tokenizer corpus documents:", len(tokenizer_corpus))

acts: 500,000 words


high_courts: 1,500,000 words


supreme_court: 1,000,000 words
tribunals: 1,000,000 words
Tokenizer corpus documents: 3574


In [12]:
BPE_VOCAB_SIZE = 20000

# Always create a fresh instance when retraining.
tokenizer = BytelevelBPE(vocab_size=BPE_VOCAB_SIZE)
tokenizer.train(tokenizer_corpus)

assert [tokenizer.stoi[s] for s in SPECIALS] == [0, 1, 2, 3]
assert tokenizer.get_vocab_size() == len(tokenizer.stoi)

examples = [
    "Section 12(1): The Court dismissed the appeal.",
    "भारत का सर्वोच्च न्यायालय",
    "₹500 — café",
    "Multiple   spaces\nand\ttabs",
]

for text in examples:
    ids = tokenizer.encode(text)

    assert all(0 <= i < tokenizer.get_vocab_size() for i in ids)
    assert tokenizer.stoi["<unk>"] not in ids
    assert tokenizer.decode(ids) == " ".join(text.split())

print("Tokenizer checks passed.")
print("Actual vocabulary:", tokenizer.get_vocab_size())

del tokenizer_corpus

Building word frequencies...


Unique whitespace-separated words: 123,357


Building initial pair statistics...


Initial unique pairs: 5,460
Target merges: 19,739



Tokenizer training complete.
Final vocabulary size: 20000
Number of merges: 19739
Tokenizer checks passed.
Actual vocabulary: 20000


## 6. Create Train and Validation Tokens Set

In [13]:
def write_token_mixture(sources, budgets, output_path):
    total_tokens = sum(budgets.values())

    token_array = np.memmap(
        output_path,
        dtype=np.uint32,
        mode="w+",
        shape=(total_tokens,),
    )

    eos_id = tokenizer.stoi["</s>"]
    vocab_size = tokenizer.get_vocab_size()

    position = 0
    source_ranges = {}

    try:
        for name, target in budgets.items():
            start = position
            written = 0
            documents_used = 0

            print(f"\nEncoding {name}: target {target:,} tokens")

            for text in sources[name]:
                remaining = target - written

                if remaining == 0:
                    break

                ids = tokenizer.encode(text)
                ids.append(eos_id)

                # The final document may be truncated at the token budget.
                chunk = ids[:remaining]

                if not all(0 <= token_id < vocab_size for token_id in chunk):
                    raise ValueError(f"{name}: invalid token ID.")

                count = len(chunk)
                token_array[position:position + count] = np.asarray(
                    chunk, dtype=np.uint32
                )

                position += count
                written += count
                documents_used += 1

            if written != target:
                raise ValueError(
                    f"{name}: needs {target:,} tokens, "
                    f"but only {written:,} are available. "
                    "No documents were repeated."
                )

            source_ranges[name] = {
                "start": start,
                "end": position,
                "tokens": written,
                "documents_used": documents_used,
            }

            print(f"Finished {name}: {written:,} tokens")

        token_array.flush()

    except Exception:
        del token_array
        if os.path.exists(output_path):
            os.remove(output_path)
        raise

    del token_array
    return source_ranges


TRAIN_IDS_PATH = os.path.join(OUTPUT_DIR, "train_ids.uint32")
VAL_IDS_PATH = os.path.join(OUTPUT_DIR, "val_ids.uint32")

train_ranges = write_token_mixture(
    train_sources, train_budgets, TRAIN_IDS_PATH
)

val_ranges = write_token_mixture(
    val_sources, val_budgets, VAL_IDS_PATH
)


Encoding acts: target 100,000,000 tokens


Finished acts: 100,000,000 tokens

Encoding high_courts: target 300,000,000 tokens


Finished high_courts: 300,000,000 tokens

Encoding supreme_court: target 200,000,000 tokens


Finished supreme_court: 200,000,000 tokens

Encoding tribunals: target 200,000,000 tokens


Finished tribunals: 200,000,000 tokens

Encoding acts: target 1,000,000 tokens


Finished acts: 1,000,000 tokens

Encoding high_courts: target 3,000,000 tokens


Finished high_courts: 3,000,000 tokens

Encoding supreme_court: target 2,000,000 tokens


Finished supreme_court: 2,000,000 tokens

Encoding tribunals: target 2,000,000 tokens


Finished tribunals: 2,000,000 tokens


In [14]:
train_ids = np.memmap(
    TRAIN_IDS_PATH, dtype=np.uint32, mode="r"
)

val_ids = np.memmap(
    VAL_IDS_PATH, dtype=np.uint32, mode="r"
)

assert len(train_ids) == TRAIN_TOKEN_BUDGET
assert len(val_ids) == VAL_TOKEN_BUDGET

metadata = {
    "seed": SEED,
    "source_ratios": SOURCE_RATIOS,
    "validation_document_fraction": VALIDATION_FRACTION,
    "tokenizer_training_words": TOKENIZER_WORD_BUDGET,
    "vocab_size": tokenizer.get_vocab_size(),
    "dtype": "uint32",
    "train": train_ranges,
    "validation": val_ranges,
}

with open(
    os.path.join(OUTPUT_DIR, "data_metadata.json"),
    "w",
    encoding="utf-8",
) as f:
    json.dump(metadata, f, indent=2)

print("Training tokens:", f"{len(train_ids):,}")
print("Validation tokens:", f"{len(val_ids):,}")
print("First IDs:", train_ids[:20])

Training tokens: 800,000,000
Validation tokens: 8,000,000
First IDs: [ 3743  4400  8613  6034  5883  1999    44  7175   291 10324  5356   641
  5917 10515   576   459   472   276   739  2038]


## 7. Create Token Blocks

In [15]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


class TokenDataset(Dataset):
    def __init__(self, path, seq_len):
        self.path = path
        self.seq_len = seq_len
        self.ids = np.memmap(path, dtype=np.uint32, mode="r")

    def __len__(self):
        return len(self.ids) // self.seq_len

    def __getitem__(self, idx):
        start = idx * self.seq_len

        # Copy only this block and convert to torch.long.
        input_ids = torch.from_numpy(
            np.array(
                self.ids[start:start + self.seq_len],
                dtype=np.int64,
                copy=True,
            )
        )

        return {
            "input_ids": input_ids,
            "labels": input_ids.clone(),
        }

## 8. Create Datasets

In [16]:
train_dataset = TokenDataset(TRAIN_IDS_PATH, TRAIN_BLOCK_SIZE)
val_dataset = TokenDataset(VAL_IDS_PATH, TRAIN_BLOCK_SIZE)

In [17]:
# Batch size 1 is only for checking the data pipeline.
check_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
)

batch = next(iter(check_loader))

print("Training blocks:", len(train_dataset))
print("Validation blocks:", len(val_dataset))
print("Input shape:", batch["input_ids"].shape)
print("Label shape:", batch["labels"].shape)
print("Dtype:", batch["input_ids"].dtype)

assert batch["input_ids"].dtype == torch.long
assert torch.equal(batch["input_ids"], batch["labels"])
assert batch["input_ids"].min().item() >= 0
assert batch["input_ids"].max().item() < tokenizer.get_vocab_size()

Training blocks: 781250
Validation blocks: 7812
Input shape: torch.Size([1, 1024])
Label shape: torch.Size([1, 1024])
Dtype: torch.int64


## 9. Define The Model

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        if self.d_head % 2 != 0:
            raise ValueError("RoPE requires an even head dimension.")

        inv_freq = 1.0 / (
            10000.0 ** (torch.arange(0, self.d_head, 2).float() / self.d_head)
        )
        angles = torch.outer(torch.arange(max_seq_len).float(), inv_freq)
        self.register_buffer("rope_cos", angles.cos(), persistent=False)
        self.register_buffer("rope_sin", angles.sin(), persistent=False)
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def apply_rope(self, q, k, offset=0):
        T = q.size(2)
        cos = self.rope_cos[offset:offset + T][None, None].to(q.dtype)
        sin = self.rope_sin[offset:offset + T][None, None].to(q.dtype)

        def rotate(x):
            first, second = x.chunk(2, dim=-1)
            return torch.cat(
                [first * cos - second * sin, first * sin + second * cos],
                dim=-1,
            )

        return rotate(q), rotate(k)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q, k = self.apply_rope(q, k)

        scores = q @ k.transpose(-2, -1) * self.attention_scale

        scores = scores.masked_fill(self.causal_mask[:T, :T] == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        weighted_sum = attn_weights @ v
        weighted_sum = weighted_sum.transpose(1, 2).contiguous().view(B, T, C)

        out = self.out_proj(weighted_sum)
        out = self.resid_dropout(out)

        return out

In [ ]:
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4          
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "rope"   
    attention_scale: float = None      
    norm_first: bool = False            # norm_first=True is for pre-norm else post-norm


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        # TODO: two linear layers with a GELU in between, plus dropout
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout,
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    """The model class the autograder imports directly. Do not rename it."""

    def __init__(self, cfg: LMConfig):
        super().__init__()
        if cfg.pos_emb_type != "rope":
            raise ValueError("This notebook requires a RoPE model/checkpoint.")
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape

        x = self.tok_emb(idx)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits


## 10. Training Arguments

In [20]:
import os
import math
import json
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VOCAB_SIZE = tokenizer.get_vocab_size()
TRAIN_BATCH_SIZE = 16

# Keep the architecture dimensions you chose.
cfg = LMConfig(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=N_EMBD,
    n_layers=N_LAYER,
    n_heads=N_HEAD,
    n_kv_heads=N_HEAD,
    d_ff=4*N_EMBD,  # Define your chosen feed-forward dimension first.
    max_seq_len=TRAIN_BLOCK_SIZE,
    pos_emb_type="rope",
    norm_first=True,
)

model = DecoderLM(cfg).to(device)

if torch.cuda.device_count() >= 2:
    model = torch.nn.DataParallel(model, device_ids=[0, 1])

# Access the underlying model for configuration and saving.
base_model = model.module if isinstance(
    model, torch.nn.DataParallel
) else model

n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params:,} parameters")

assert 50_000_000 <= n_params <= 300_000_000, (
    "Model must contain 50M–300M parameters."
)

# Use the path-based TokenDataset defined for your capstone.
train_dataset = TokenDataset(TRAIN_IDS_PATH, cfg.max_seq_len)
val_dataset = TokenDataset(VAL_IDS_PATH, cfg.max_seq_len)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

56,420,000 parameters


In [21]:
base_model = (
    model.module
    if isinstance(model, torch.nn.DataParallel)
    else model
)

# Run ONLY for a fresh, untrained model.
with torch.no_grad():
    torch.nn.init.normal_(
        base_model.tok_emb.weight,
        mean=0.0,
        std=0.02,
    )

assert base_model.head.weight is base_model.tok_emb.weight
print("Embedding std:", base_model.tok_emb.weight.std().item())

Embedding std: 0.01999848522245884


## 11. Training

In [22]:
@torch.no_grad()
def evaluate_validation(model, val_loader, device):
    model.eval()

    total_nll = 0.0
    total_targets = 0

    for batch in val_loader:
        x = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        targets = labels[:, 1:]

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=device.type == "cuda",
        ):
            logits = model(x)

            loss_mean = F.cross_entropy(
                logits[:, :-1, :].reshape(-1, logits.size(-1)),
                targets.reshape(-1),
            )

        count = targets.numel()
        total_nll += loss_mean.item() * count
        total_targets += count

    if total_targets == 0:
        raise ValueError("Validation loader is empty.")

    val_loss = total_nll / total_targets
    val_ppl = math.exp(val_loss) if val_loss < 700 else float("inf")

    return val_loss, val_ppl

In [23]:
from dataclasses import asdict


def train_model(
    model,
    train_loader,
    val_loader,
    train_steps,
    lr,
    accumulation_steps=1,
    log_every=100,
    eval_every=1000,
):
    if len(train_loader) == 0:
        raise ValueError("Training loader is empty.")

    if min(train_steps, accumulation_steps, log_every, eval_every) < 1:
        raise ValueError("Steps and intervals must be positive.")

    base_model = (
        model.module
        if isinstance(model, torch.nn.DataParallel)
        else model
    )
    device = next(base_model.parameters()).device
    use_amp = device.type == "cuda"

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    optimizer.zero_grad(set_to_none=True)

    os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
    best_path = os.path.join(MODEL_OUTPUT_DIR, "best.pt")
    latest_path = os.path.join(MODEL_OUTPUT_DIR, "latest.pt")

    loss_history = []
    step_history = []
    validation_history = []

    best_val_loss = float("inf")
    step = 0
    optimizer_steps = 0
    skipped_updates = 0

    pending_batches = 0
    pending_targets = 0
    reference_targets = None
    evaluation_due = False

    interval_nll = 0.0
    interval_targets = 0

    def save_atomic(checkpoint, path):
        temporary_path = path + ".tmp"
        torch.save(checkpoint, temporary_path)
        os.replace(temporary_path, path)

    model.train()

    while step < train_steps:
        for batch in train_loader:
            x = batch["input_ids"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            targets = labels[:, 1:]
            target_count = targets.numel()

            if target_count == 0:
                raise ValueError("Each sequence needs at least two tokens.")

            # Establish a normalization reference for this accumulation group.
            if pending_batches == 0:
                reference_targets = target_count * accumulation_steps

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=use_amp,
            ):
                logits = model(x)
                loss_mean = F.cross_entropy(
                    logits[:, :-1, :].reshape(-1, logits.size(-1)),
                    targets.reshape(-1),
                )

            if not torch.isfinite(loss_mean):
                raise RuntimeError(
                    f"Non-finite training loss at batch {step + 1}"
                )

            batch_loss = loss_mean.detach().item()
            weighted_loss = loss_mean * (
                target_count / reference_targets
            )

            scaler.scale(weighted_loss).backward()

            pending_batches += 1
            pending_targets += target_count
            step += 1

            interval_nll += batch_loss * target_count
            interval_targets += target_count

            if step == 1 or step % log_every == 0 or step == train_steps:
                average_loss = interval_nll / interval_targets
                loss_history.append(average_loss)
                step_history.append(step)

                print(
                    f"batch {step}/{train_steps} | "
                    f"average train loss={average_loss:.4f}",
                    flush=True,
                )

                interval_nll = 0.0
                interval_targets = 0

            evaluation_due |= step % eval_every == 0

            should_update = (
                pending_batches == accumulation_steps
                or step == train_steps
            )

            # Release batch activations before validation.
            del logits, loss_mean, weighted_loss, targets, x, labels

            if should_update:
                scaler.unscale_(optimizer)

                # Normalize by actual tokens, including partial final groups.
                correction = reference_targets / pending_targets
                for parameter in model.parameters():
                    if parameter.grad is not None:
                        parameter.grad.mul_(correction)

                previous_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

                update_skipped = scaler.get_scale() < previous_scale

                if update_skipped:
                    skipped_updates += 1
                    print(
                        f"AMP skipped update at batch {step}",
                        flush=True,
                    )
                else:
                    optimizer_steps += 1

                pending_batches = 0
                pending_targets = 0

                if evaluation_due or step == train_steps:
                    val_loss, val_ppl = evaluate_validation(
                        model, val_loader, device
                    )

                    validation_history.append({
                        "step": step,
                        "optimizer_steps": optimizer_steps,
                        "val_loss": val_loss,
                        "val_ppl": val_ppl,
                    })

                    if val_loss < best_val_loss:
                        best_val_loss = val_loss

                        save_atomic({
                            "model_config": asdict(base_model.cfg),
                            "model_state_dict": base_model.state_dict(),
                            "step": step,
                            "val_loss": val_loss,
                        }, best_path)

                    save_atomic({
                        "model_config": asdict(base_model.cfg),
                        "model_state_dict": base_model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "scaler_state_dict": scaler.state_dict(),
                        "step": step,
                        "optimizer_steps": optimizer_steps,
                        "skipped_updates": skipped_updates,
                        "accumulation_steps": accumulation_steps,
                        "best_val_loss": best_val_loss,
                    }, latest_path)

                    with open(
                        os.path.join(MODEL_OUTPUT_DIR, "history.json"),
                        "w",
                        encoding="utf-8",
                    ) as f:
                        json.dump({
                            "train_steps": step_history,
                            "train_losses": loss_history,
                            "validation": validation_history,
                        }, f, indent=2)

                    print(
                        f"Updates={optimizer_steps} | "
                        f"Validation loss={val_loss:.4f} | "
                        f"Perplexity={val_ppl:.2f}",
                        flush=True,
                    )

                    evaluation_due = False
                    model.train()

            if step >= train_steps:
                break

    # Restore the best evaluated weights for inference.
    checkpoint = torch.load(
        best_path,
        map_location="cpu",
        weights_only=True,
    )
    base_model.load_state_dict(checkpoint["model_state_dict"])
    del checkpoint
    model.eval()

    print(f"Batches processed: {step}")
    print(f"Successful optimizer updates: {optimizer_steps}")
    print(f"Skipped AMP updates: {skipped_updates}")
    print(f"Best validation loss: {best_val_loss:.4f}")

    return loss_history, step_history, validation_history

In [ ]:
TRAIN_STEPS = 50000
LEARNING_RATE = 3e-4
GRAD_ACCUM_STEPS = 1

loss_history, step_history, validation_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    train_steps=TRAIN_STEPS,
    lr=LEARNING_RATE,
    accumulation_steps=GRAD_ACCUM_STEPS,
    log_every=100,
    eval_every=1000,
)

## 12. Perplexity

In [ ]:
# Use the full val_loader, not the two-batch sample-check list.
val_loss, val_perplexity = evaluate_validation(
    model=model,
    val_loader=val_loader,
    device=device,
)

print(f"Validation loss: {val_loss:.6f}")
print(f"Validation perplexity: {val_perplexity:.2f}")

## 13. Plot Training and Validation Loss

In [ ]:
import matplotlib.pyplot as plt

val_steps = [record["step"] for record in validation_history]
val_losses = [record["val_loss"] for record in validation_history]

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    step_history,
    loss_history,
    marker=".",
    label="Training loss (logged batch)",
)

ax.plot(
    val_steps,
    val_losses,
    marker="o",
    linestyle="--",
    label="Validation loss (average)",
)

ax.set_xlabel("Training step")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Training and Validation Loss")
ax.grid(alpha=0.3)
ax.legend()

fig.tight_layout()
fig.savefig(LOSS_PLOT_PATH, dpi=200)
plt.show()

print("Plot saved:", LOSS_PLOT_PATH)

## 14. Save Final Model to Kaggle Output

In [ ]:
from dataclasses import asdict
import os
import torch

base_model = (
    model.module
    if isinstance(model, torch.nn.DataParallel)
    else model
)

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

model_path = os.path.join(MODEL_OUTPUT_DIR, "model.pt")

torch.save(
    {
        "model_config": asdict(base_model.cfg),
        "model_state_dict": {
            name: tensor.detach().cpu()
            for name, tensor in base_model.state_dict().items()
        },
    },
    model_path,
)

print("Model saved:", model_path)

In [ ]:
import os
import json

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

tokenizer_state = {
    "vocab_size": tokenizer.get_vocab_size(),
    "specials": tokenizer.specials,
    "vocab": tokenizer.vocab,
    "merges": [list(pair) for pair in tokenizer.merges],
    "merge_ranks": [
        {
            "pair": list(pair),
            "rank": rank,
            "merged_symbol": merged_symbol,
        }
        for pair, (rank, merged_symbol)
        in tokenizer.merge_ranks.items()
    ],
    "symbol_to_bytes": {
        symbol: list(value)
        for symbol, value in tokenizer.symbol_to_bytes.items()
    },
}

tokenizer_path = os.path.join(MODEL_OUTPUT_DIR, "tokenizer.json")
temporary_path = tokenizer_path + ".tmp"

with open(temporary_path, "w", encoding="utf-8") as f:
    json.dump(tokenizer_state, f, ensure_ascii=False)

os.replace(temporary_path, tokenizer_path)

print("Tokenizer saved:", tokenizer_path)

In [ ]:
import pickle

history = {
    "loss_history": loss_history,
    "step_history": step_history,
    "validation_history": validation_history,
}

with open(
    os.path.join(OUTPUT_DIR, "legal-gpt2-final", "history.pkl"), "wb"
) as f:
    pickle.dump(history, f)

In [ ]:
import shutil
from IPython.display import FileLink, display

zip_path = shutil.make_archive(
    os.path.join(OUTPUT_DIR, "model_files"),
    "zip",
    root_dir=os.path.join(OUTPUT_DIR, "legal-gpt2-final"),
)

display(FileLink(zip_path))

## 15. Push Model to Hugging Face Hub

In [ ]:
# from huggingface_hub import HfApi

# # Replace with your Hugging Face username and repository name.
# HF_MODEL_REPO = "YOUR_USERNAME/cs781-phase1"

# api = HfApi(token=HF_TOKEN)

# api.create_repo(
#     repo_id=HF_MODEL_REPO,
#     repo_type="model",
#     private=True,
#     exist_ok=True,
# )

# for filename in ["model.pt", "tokenizer.json"]:
#     path = os.path.join(MODEL_OUTPUT_DIR, filename)

#     if not os.path.isfile(path):
#         raise FileNotFoundError(path)

#     api.upload_file(
#         path_or_fileobj=path,
#         path_in_repo=filename,
#         repo_id=HF_MODEL_REPO,
#         repo_type="model",
#     )

# print("Uploaded model and tokenizer.")
# print(f"https://huggingface.co/{HF_MODEL_REPO}")

## 16. Load Test Data from HuggingFace

In [ ]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:", test_file_path)

## 17. Load Test Prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

assert isinstance(test_data, dict), "Expected a dictionary of test IDs."

for example_id, record in test_data.items():
    assert isinstance(record, dict)
    assert isinstance(record.get("text"), str), (
        f"Invalid prompt for ID {example_id}"
    )

print("Test examples:", len(test_data))

## 18. Load Final Model for Generation

In [ ]:
MODEL_OUTPUT_DIR = "/kaggle/input/datasets/tejasvaidya137/saved-files-model1"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    os.path.join(MODEL_OUTPUT_DIR, "model.pt"),
    map_location="cpu",
    weights_only=True,
)

generation_cfg = LMConfig(**checkpoint["model_config"])
generation_model = DecoderLM(generation_cfg)

generation_model.load_state_dict(checkpoint["model_state_dict"])
del checkpoint

generation_model = generation_model.to(device).eval()

with open(
    os.path.join(MODEL_OUTPUT_DIR, "tokenizer.json"),
    "r",
    encoding="utf-8",
) as f:
    state = json.load(f)

generation_tokenizer = BytelevelBPE(vocab_size=state["vocab_size"])
generation_tokenizer.specials = state["specials"]
generation_tokenizer.vocab = state["vocab"]
generation_tokenizer.vocab_size = len(state["vocab"])

generation_tokenizer.stoi = {
    symbol: i for i, symbol in enumerate(state["vocab"])
}
generation_tokenizer.itos = {
    i: symbol for symbol, i in generation_tokenizer.stoi.items()
}

generation_tokenizer.merges = [
    tuple(pair) for pair in state["merges"]
]
generation_tokenizer.merge_ranks = {
    tuple(record["pair"]): (
        record["rank"],
        record["merged_symbol"],
    )
    for record in state["merge_ranks"]
}
generation_tokenizer.symbol_to_bytes = {
    symbol: bytes(values)
    for symbol, values in state["symbol_to_bytes"].items()
}

assert generation_cfg.vocab_size == generation_tokenizer.get_vocab_size()

print("Model and tokenizer loaded.")

## 19. Generate 128 Words
#### (words, not tokens)

In [ ]:
@torch.inference_mode()
def cached_forward(model, idx, past=None):
    """Return last-position logits and one (K, V) cache per layer."""
    if model.cfg.pos_emb_type != "rope":
        raise NotImplementedError("This helper supports RoPE.")

    if past is None:
        past = [None] * len(model.blocks)

    if len(past) != len(model.blocks):
        raise ValueError("Expected one cache entry per block.")

    offset = 0 if past[0] is None else past[0][0].size(2)
    B, T = idx.shape

    if offset + T > model.cfg.max_seq_len:
        raise ValueError("Cache exceeds model context length.")

    x = model.tok_emb(idx)
    updated_cache = []

    for block, previous in zip(model.blocks, past):
        attention = block.attn
        attn_input = block.ln1(x) if block.norm_first else x

        q = attention.q_proj(attn_input).view(
            B, T, attention.n_heads, attention.d_head
        ).transpose(1, 2)

        k = attention.k_proj(attn_input).view(
            B, T, attention.n_heads, attention.d_head
        ).transpose(1, 2)

        v = attention.v_proj(attn_input).view(
            B, T, attention.n_heads, attention.d_head
        ).transpose(1, 2)

        # Only rotate new keys; cached keys are already rotated.
        q, k = attention.apply_rope(q, k, offset=offset)

        if previous is not None:
            k = torch.cat([previous[0], k], dim=2)
            v = torch.cat([previous[1], v], dim=2)

        scores = (q @ k.transpose(-2, -1)) * attention.attention_scale

        # Query positions begin after the cached prefix.
        mask = attention.causal_mask[
            offset:offset + T, :offset + T
        ]
        scores = scores.masked_fill(~mask, float("-inf"))

        weights = torch.softmax(scores, dim=-1)
        weights = attention.attn_dropout(weights)

        output = (weights @ v).transpose(1, 2).contiguous().view(
            B, T, model.cfg.d_model
        )
        output = attention.resid_dropout(attention.out_proj(output))

        if block.norm_first:
            x = x + output
            x = x + block.ffn(block.ln2(x))
        else:
            x = block.ln1(x + output)
            x = block.ln2(x + block.ffn(x))

        updated_cache.append((k, v))

    # Generation only needs logits for the last position.
    logits = model.head(model.ln_f(x[:, -1:, :]))
    return logits, updated_cache

In [ ]:
@torch.inference_mode()
def generate_continuation(
    prompt,
    model,
    tokenizer,
    device,
    target_words=128,
    max_new_tokens=1024,
    temperature=1.0,
    top_k=1
):
    if temperature <= 0:
        raise ValueError("temperature must be positive")
    
    if not isinstance(top_k, int) or top_k < 0:
        raise ValueError("top_k must be a nonnegative integer")
    
    model.eval()

    prompt_ids = tokenizer.encode(prompt)
    if not prompt_ids:
        raise ValueError("Prompt contains no tokens.")

    context_limit = model.cfg.max_seq_len

    context = torch.tensor(
        [prompt_ids[-context_limit:]],
        dtype=torch.long,
        device=device,
    )

    # Count word boundaries efficiently during generation.
    ends_word = [
        tokenizer._symbol_ends_word(tokenizer.itos[i])
        for i in range(tokenizer.get_vocab_size())
    ]

    special_ids = list(tokenizer.stoi[s] for s in tokenizer.specials)

    generated_ids = []
    completed_words = 0

    past = None
    model_input = context

    MAX_TOKENS_PER_WORD = 16
    MAX_TOKENS_PER_WORD = 16
    tokens_since_boundary = 0
    bytes_since_boundary = 0
    
    word_end_id = tokenizer.stoi["</w>"]
    
    empty_token_ids = [
        i
        for i in range(tokenizer.get_vocab_size())
        if i not in special_ids
        and not tokenizer.symbol_to_bytes.get(tokenizer.itos[i], b"")
    ]

    for _ in range(max_new_tokens):
        outputs, past = cached_forward(model, model_input, past)
        logits = outputs[:, -1, :].clone()

        # Require a fixed-length continuation without special tokens.
        logits[:, special_ids] = float("-inf")

        # Prevent empty words.
        if bytes_since_boundary == 0:
            logits[:, empty_token_ids] = float("-inf")
        
        # End an unusually long word using the tokenizer's word marker.
        if (
            tokens_since_boundary >= MAX_TOKENS_PER_WORD
            and bytes_since_boundary > 0
        ):
            logits.fill_(float("-inf"))
            logits[:, word_end_id] = 0.0

        # Special-token masking remains above this section.
        logits = logits / temperature
        
        if top_k > 0:
            allowed_count = logits.size(-1) - len(special_ids)
            k = min(top_k, allowed_count)
        
            top_logits, top_ids = torch.topk(logits, k=k, dim=-1)
            probabilities = torch.softmax(top_logits, dim=-1)
        
            selected = torch.multinomial(probabilities, num_samples=1)
            next_id = int(top_ids.gather(-1, selected).item())
        else:
            probabilities = torch.softmax(logits, dim=-1)
            next_id = int(
                torch.multinomial(probabilities, num_samples=1).item()
            )
        generated_ids.append(next_id)

        tokens_since_boundary += 1

        symbol = tokenizer.itos[next_id]
        bytes_since_boundary += len(
            tokenizer.symbol_to_bytes.get(symbol, b"")
        )
        
        if ends_word[next_id]:
            if bytes_since_boundary > 0:
                completed_words += 1
        
            tokens_since_boundary = 0
            bytes_since_boundary = 0

        if completed_words >= target_words:
            text = tokenizer.decode(generated_ids)
            words = text.split()

            if len(words) >= target_words:
                return " ".join(words[:target_words])

        next_token = torch.tensor(
            [[next_id]], dtype=torch.long, device=device
        )

        context = torch.cat([context, next_token], dim=1)

        if context.size(1) > context_limit:
            # Rebuild after cropping to preserve your original sliding-window behavior.
            context = context[:, -context_limit:]
            past = None
            model_input = context
        else:
            # Only process the newly generated token next time.
            model_input = next_token

    raise RuntimeError(
        f"Did not generate {target_words} words within "
        f"{max_new_tokens} tokens. Completed boundaries: {completed_words}."
    )

## 20. Generate Predictions for the Test Set

In [ ]:
TOP_K = 70

In [ ]:
import os
import json
import time

PROGRESS_PATH = os.path.join(OUTPUT_DIR, "generation_progress.json")

# Resume predictions from this same model and decoding configuration.
if os.path.isfile(PROGRESS_PATH):
    with open(PROGRESS_PATH, "r", encoding="utf-8") as f:
        predictions = json.load(f)
else:
    predictions = {}

# Check saved progress before resuming.
assert set(predictions).issubset(test_data), "Unknown IDs in progress file."

for example_id, record in predictions.items():
    assert isinstance(record.get("text"), str)
    assert len(record["text"].split()) == 128, (
        f"Invalid saved continuation: {example_id}"
    )

total = len(test_data)
start_time = time.time()

for example_id, record in test_data.items():
    if example_id in predictions:
        continue

    continuation = generate_continuation(
        prompt=record["text"],
        model=generation_model,
        tokenizer=generation_tokenizer,
        device=device,
        target_words=128,
        max_new_tokens=4096,
        temperature=TEMPERATURE,
        top_k=TOP_K,
    )

    assert len(continuation.split()) == 128

    predictions[example_id] = {"text": continuation}

    # Write atomically to preserve the previous progress file.
    temporary_path = PROGRESS_PATH + ".tmp"

    with open(temporary_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)

    os.replace(temporary_path, PROGRESS_PATH)

    print(
        f"Completed {len(predictions)}/{total} | "
        f"ID: {example_id} | "
        f"Elapsed: {(time.time() - start_time) / 60:.1f} min",
        flush=True,
    )

assert set(predictions) == set(test_data)
assert all(
    len(record["text"].split()) == 128
    for record in predictions.values()
)

print("All predictions generated and validated.")

## 21. Save prediction.json

In [ ]:
import os
import json

assert set(predictions) == set(test_data), (
    "Prediction IDs do not match test IDs."
)

for example_id, record in predictions.items():
    assert isinstance(record, dict)
    assert isinstance(record.get("text"), str)
    assert len(record["text"].split()) == 128, (
        f"ID {example_id}: expected 128 words, "
        f"got {len(record['text'].split())}"
    )

PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print("Saved:", PREDICTION_PATH)
print("Predictions:", len(predictions))

## 22. Create CodaBench Submission ZIP

In [ ]:
import zipfile

SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")

with zipfile.ZipFile(
    SUBMISSION_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        PREDICTION_PATH,
        arcname="prediction.json",
    )

# Verify the archive structure and saved contents.
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as archive:
    assert archive.namelist() == ["prediction.json"]
    assert archive.testzip() is None

    saved_predictions = json.loads(
        archive.read("prediction.json").decode("utf-8")
    )
    assert saved_predictions == predictions

print("Submission ZIP verified:", SUBMISSION_ZIP)